In [1]:
pip install -q torch transformers datasets accelerate huggingface_hub pandas matplotlib

In [ ]:
#!/usr/bin/env python
"""
Evaluate PKU-SafeRLHF-based sequence-classifier reward models on RewardBench
"""

import os
import json
import time
import random
from typing import List, Tuple

import numpy as np
import pandas as pd
import torch

from getpass import getpass
from datasets import load_dataset, Dataset
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from google.colab import drive

drive.mount("/content/drive")

# =========================================================
# CONFIG
# =========================================================

HF_TOKEN = getpass("Enter your Hugging Face token, or press Enter if public: ").strip()

if HF_TOKEN:
    try:
        login(token=HF_TOKEN)
        print("Hugging Face login successful.")
    except Exception as e:
        print(f"HF login warning: {e}")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_LENGTH = 512
BATCH_SIZE = 16

# ---------------------------------------------------------
# Multiple seeds
# ---------------------------------------------------------

SEEDS = [188,144]

# ---------------------------------------------------------
# Evaluation mode
# ---------------------------------------------------------

USE_SUBSET = True
TOTAL_SAMPLES = 500

# ---------------------------------------------------------
# PKU-SafeRLHF reward models to compare
# ---------------------------------------------------------
# IMPORTANT:
# If your semantic-MARS repo name differs, update only the last repo ID.

MODELS: List[Tuple[str, str]] = [
    ("AdaBoost", ""),
    ("No aug.", ""),
    ("Uniform aug.", ""),
    ("WoN", ""),
    ("MARS", ""),
]
DATASET_NAME = "allenai/reward-bench"
DATASET_SPLIT = "raw"

OUT_DIR = ""
os.makedirs(OUT_DIR, exist_ok=True)

# =========================================================
# OFFICIAL-STYLE REWARDBENCH GROUPING
# =========================================================

SECTION_MAP = {
    "Chat": [
        "alpacaeval-easy",
        "alpacaeval-length",
        "alpacaeval-hard",
        "mt-bench-easy",
        "mt-bench-med",
    ],
    "Chat Hard": [
        "mt-bench-hard",
        "llmbar-natural",
        "llmbar-adver-neighbor",
        "llmbar-adver-GPTInst",
        "llmbar-adver-GPTOut",
        "llmbar-adver-manual",
    ],
    "Safety": [
        "refusals-dangerous",
        "refusals-offensive",
        "xstest-should-refuse",
        "xstest-should-respond",
        "donotanswer",
    ],
    "Reasoning": [
        "math-prm",
        "hep-cpp",
        "hep-go",
        "hep-java",
        "hep-js",
        "hep-python",
        "hep-rust",
    ],
}

SECTION_COLUMNS = [
    "Chat",
    "Chat Hard",
    "Safety",
    "Reasoning",
    "RewardBench Core",
]

# =========================================================
# REPRODUCIBILITY
# =========================================================

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# =========================================================
# HELPERS
# =========================================================

def ensure_pad_token(tokenizer):
    if tokenizer.pad_token is None:
        if tokenizer.eos_token is not None:
            tokenizer.pad_token = tokenizer.eos_token
        else:
            tokenizer.add_special_tokens({"pad_token": "[PAD]"})


def load_rm(model_name_or_path: str):
    common_kwargs = {}

    if HF_TOKEN:
        common_kwargs["token"] = HF_TOKEN

    tokenizer = AutoTokenizer.from_pretrained(
        model_name_or_path,
        use_fast=True,
        **common_kwargs,
    )

    ensure_pad_token(tokenizer)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name_or_path,
        **common_kwargs,
    )

    if len(tokenizer) != model.get_input_embeddings().num_embeddings:
        model.resize_token_embeddings(len(tokenizer))

    model.to(DEVICE)
    model.eval()

    return tokenizer, model


def logits_to_scores(logits: np.ndarray) -> np.ndarray:
    logits = np.asarray(logits)

    if logits.ndim == 1:
        return logits.astype(np.float32)

    if logits.ndim == 2:
        if logits.shape[1] == 1:
            return logits[:, 0].astype(np.float32)

        return logits[:, -1].astype(np.float32)

    raise ValueError(f"Unexpected logits shape: {logits.shape}")


def looks_like_full_dialogue(text: str) -> bool:
    if text is None:
        return False

    t = str(text).strip()

    if len(t) < 20:
        return False

    return (
        "Human:" in t[:200]
        or "Assistant:" in t[:200]
        or "User:" in t[:200]
    )


def build_pair_text(prompt: str, answer: str) -> str:
    answer = "" if answer is None else str(answer)

    if looks_like_full_dialogue(answer):
        return answer

    prompt = "" if prompt is None else str(prompt)

    return f"Human: {prompt}\n\nAssistant: {answer}"


@torch.no_grad()
def score_texts(tokenizer, model, texts: List[str], batch_size: int = BATCH_SIZE) -> np.ndarray:
    all_scores = []

    for start in range(0, len(texts), batch_size):
        end = min(start + batch_size, len(texts))
        batch = texts[start:end]

        enc = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        )

        enc = {k: v.to(DEVICE) for k, v in enc.items()}

        out = model(**enc)
        batch_scores = logits_to_scores(
            out.logits.detach().float().cpu().numpy()
        )

        all_scores.append(batch_scores)

    return np.concatenate(all_scores, axis=0)


def load_rewardbench():
    ds = load_dataset(DATASET_NAME, split=DATASET_SPLIT)

    required = {"prompt", "chosen", "rejected"}
    missing = required - set(ds.column_names)

    if missing:
        raise ValueError(
            f"Missing required columns: {missing}. Found: {ds.column_names}"
        )

    subset_col = None

    for c in ["subset", "subsubset", "eval_set", "source"]:
        if c in ds.column_names:
            subset_col = c
            break

    if subset_col is None:
        raise ValueError(
            f"Could not find subset/category column. Found: {ds.column_names}"
        )

    return ds, subset_col


def normalize_subset_name(x: str) -> str:
    x = str(x).strip()

    aliases = {
        "AlpacaEval Easy": "alpacaeval-easy",
        "AlpacaEval Length": "alpacaeval-length",
        "AlpacaEval Hard": "alpacaeval-hard",
        "MT Bench Easy": "mt-bench-easy",
        "MT Bench Medium": "mt-bench-med",
        "MT Bench Hard": "mt-bench-hard",
        "LLMBar Natural": "llmbar-natural",
        "LLMBar Adver. Neighbor": "llmbar-adver-neighbor",
        "LLMBar Adver. GPTInst": "llmbar-adver-GPTInst",
        "LLMBar Adver. GPTOut": "llmbar-adver-GPTOut",
        "LLMBar Adver. Manual": "llmbar-adver-manual",
        "Refusals Dangerous": "refusals-dangerous",
        "Refusals Offensive": "refusals-offensive",
        "XSTest Should Refuse": "xstest-should-refuse",
        "XSTest Should Respond": "xstest-should-respond",
        "Do Not Answer": "donotanswer",
        "PRM Math": "math-prm",
        "HumanEvalPack CPP": "hep-cpp",
        "HumanEvalPack Go": "hep-go",
        "HumanEvalPack Java": "hep-java",
        "HumanEvalPack Javascript": "hep-js",
        "HumanEvalPack Python": "hep-python",
        "HumanEvalPack Rust": "hep-rust",
    }

    return aliases.get(x, x)


def stratified_sample(ds, subset_col, total_samples=500, seed=42):
    df = ds.to_pandas().copy()
    df["_subset_norm"] = df[subset_col].astype(str)

    subset_counts = df["_subset_norm"].value_counts().sort_index()
    total_available = len(df)

    if total_samples >= total_available:
        print(
            f"Requested {total_samples} >= dataset size {total_available}; using full dataset."
        )
        return df.drop(columns=["_subset_norm"]).reset_index(drop=True)

    raw_alloc = (subset_counts / total_available) * total_samples
    alloc = np.floor(raw_alloc).astype(int)

    if total_samples >= len(subset_counts):
        alloc[alloc == 0] = 1

    current_total = int(alloc.sum())

    if current_total > total_samples:
        excess = current_total - total_samples

        for s in alloc.sort_values(ascending=False).index:
            while excess > 0 and alloc[s] > 1:
                alloc[s] -= 1
                excess -= 1

            if excess == 0:
                break

    elif current_total < total_samples:
        deficit = total_samples - current_total
        remainders = (raw_alloc - np.floor(raw_alloc)).sort_values(ascending=False)
        order = list(remainders.index)
        i = 0

        while deficit > 0:
            s = order[i % len(order)]

            if alloc[s] < subset_counts[s]:
                alloc[s] += 1
                deficit -= 1

            i += 1

    sampled_parts = []

    for subset_name, n_take in alloc.items():
        part = df[df["_subset_norm"] == subset_name]
        n_take = min(int(n_take), len(part))

        if n_take > 0:
            sampled_parts.append(part.sample(n=n_take, random_state=seed))

    sampled_df = (
        pd.concat(sampled_parts, axis=0)
        .sample(frac=1.0, random_state=seed)
        .reset_index(drop=True)
    )

    return sampled_df.drop(columns=["_subset_norm"])


def compute_section_scores(per_example: pd.DataFrame):
    subset_df = (
        per_example.groupby("subset", as_index=False)
        .agg(
            n_prompts=("win", "size"),
            pairwise_accuracy=("win", "mean"),
            avg_margin=("margin", "mean"),
        )
        .sort_values("subset")
        .reset_index(drop=True)
    )

    subset_to_acc = dict(
        zip(subset_df["subset"], subset_df["pairwise_accuracy"])
    )

    subset_to_n = dict(
        zip(subset_df["subset"], subset_df["n_prompts"])
    )

    section_scores = {}

    for section_name, subset_list in SECTION_MAP.items():
        numer = 0.0
        denom = 0

        for s in subset_list:
            if s in subset_to_acc and s in subset_to_n:
                numer += subset_to_acc[s] * subset_to_n[s]
                denom += subset_to_n[s]

        section_scores[section_name] = (
            float(numer / denom) if denom > 0 else float("nan")
        )

    valid_scores = [
        v for v in section_scores.values()
        if not np.isnan(v)
    ]

    section_scores["RewardBench Core"] = (
        float(np.mean(valid_scores)) if valid_scores else float("nan")
    )

    return subset_df, section_scores


def evaluate_one_model(tag: str, repo_id: str, ds, subset_col: str, seed: int):
    print(f"\n=== Evaluating {tag}: {repo_id} | seed={seed} ===")
    model_start = time.time()

    tokenizer, model = load_rm(repo_id)

    chosen_texts = [
        build_pair_text(p, c)
        for p, c in zip(ds["prompt"], ds["chosen"])
    ]

    rejected_texts = [
        build_pair_text(p, r)
        for p, r in zip(ds["prompt"], ds["rejected"])
    ]

    t0 = time.time()

    chosen_scores = score_texts(
        tokenizer,
        model,
        chosen_texts,
        batch_size=BATCH_SIZE,
    )

    t1 = time.time()

    rejected_scores = score_texts(
        tokenizer,
        model,
        rejected_texts,
        batch_size=BATCH_SIZE,
    )

    t2 = time.time()

    margins = chosen_scores - rejected_scores
    wins = (margins > 0).astype(np.int32)

    per_example = pd.DataFrame({
        "seed": seed,
        "model_tag": tag,
        "model_name": repo_id,
        "subset_raw": ds[subset_col],
        "subset": [normalize_subset_name(x) for x in ds[subset_col]],
        "chosen_score": chosen_scores,
        "rejected_score": rejected_scores,
        "margin": margins,
        "win": wins,
    })

    subset_df, section_scores = compute_section_scores(per_example)

    summary = {
        "seed": seed,
        "model_tag": tag,
        "model_name": repo_id,
        "n_examples": len(per_example),
        "chosen_pass_sec": round(t1 - t0, 2),
        "rejected_pass_sec": round(t2 - t1, 2),
        "total_eval_sec": round(time.time() - model_start, 2),
        **section_scores,
    }

    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return per_example, subset_df, summary


def aggregate_mean_std(summary_df: pd.DataFrame):
    rows = []

    for model_tag, grp in summary_df.groupby("model_tag"):
        row = {
            "model_tag": model_tag,
            "model_name": grp["model_name"].iloc[0],
            "n_seeds": grp["seed"].nunique(),
            "n_examples_per_seed": int(grp["n_examples"].iloc[0]),
        }

        for col in SECTION_COLUMNS:
            row[f"{col}_mean"] = float(grp[col].mean())
            row[f"{col}_std"] = float(grp[col].std(ddof=0))

        rows.append(row)

    out = pd.DataFrame(rows)

    out = out.sort_values(
        "RewardBench Core_mean",
        ascending=False,
    ).reset_index(drop=True)

    return out


def main():
    overall_start = time.time()

    print(f"Device: {DEVICE}")
    print(f"torch version: {torch.__version__}")
    print(f"cuda available: {torch.cuda.is_available()}")
    print(f"Seeds: {SEEDS}")

    ds_full, subset_col = load_rewardbench()

    print(f"Loaded {DATASET_NAME} [{DATASET_SPLIT}] with {len(ds_full)} examples")
    print(f"Using subset column: {subset_col}")

    all_per_example = []
    all_subset_scores = []
    all_summaries = []
    all_failures = []

    for seed in SEEDS:
        print("\n" + "=" * 100)
        print(f"RUNNING SEED = {seed}")
        print("=" * 100)

        set_seed(seed)

        if USE_SUBSET:
            print(f"\nUsing stratified subset with TOTAL_SAMPLES={TOTAL_SAMPLES}")
            sampled_df = stratified_sample(
                ds_full,
                subset_col,
                total_samples=TOTAL_SAMPLES,
                seed=seed,
            )

            ds_seed = Dataset.from_pandas(
                sampled_df,
                preserve_index=False,
            )

            print(f"Subset size after stratified sampling: {len(ds_seed)}")

        else:
            print("\nUsing full RewardBench dataset")
            ds_seed = ds_full

        for tag, repo_id in MODELS:
            try:
                per_example, subset_df, summary = evaluate_one_model(
                    tag=tag,
                    repo_id=repo_id,
                    ds=ds_seed,
                    subset_col=subset_col,
                    seed=seed,
                )

                subset_df = subset_df.copy()
                subset_df.insert(0, "seed", seed)
                subset_df.insert(1, "model_tag", tag)
                subset_df.insert(2, "model_name", repo_id)

                all_per_example.append(per_example)
                all_subset_scores.append(subset_df)
                all_summaries.append(summary)

            except Exception as e:
                msg = str(e)

                all_failures.append({
                    "seed": seed,
                    "model_tag": tag,
                    "model_name": repo_id,
                    "error": msg,
                })

                print(f"\n[FAILED] seed={seed} | {tag}: {repo_id}")
                print(msg)

    if all_summaries:
        summary_df = pd.DataFrame(all_summaries)

        per_seed_summary_path = os.path.join(
            OUT_DIR,
            "all_models_section_summary_per_seed.csv",
        )

        summary_df.to_csv(per_seed_summary_path, index=False)

        agg_df = aggregate_mean_std(summary_df)

        agg_path = os.path.join(
            OUT_DIR,
            "all_models_section_summary_mean_std.csv",
        )

        agg_df.to_csv(agg_path, index=False)

        if all_subset_scores:
            all_subset_df = pd.concat(
                all_subset_scores,
                ignore_index=True,
            )

            all_subset_df.to_csv(
                os.path.join(OUT_DIR, "all_models_subset_scores_all_seeds.csv"),
                index=False,
            )

        if all_per_example:
            all_per_example_df = pd.concat(
                all_per_example,
                ignore_index=True,
            )

            all_per_example_df.to_csv(
                os.path.join(OUT_DIR, "all_models_per_example_all_seeds.csv"),
                index=False,
            )

        print("\n================ PER-SEED SUMMARY ================\n")
        print(
            summary_df
            .sort_values(["seed", "RewardBench Core"], ascending=[True, False])
            .to_string(index=False)
        )

        print("\n================ MEAN ± STD SUMMARY ================\n")

        display_cols = [
            "model_tag",
            "Chat_mean",
            "Chat_std",
            "Chat Hard_mean",
            "Chat Hard_std",
            "Safety_mean",
            "Safety_std",
            "Reasoning_mean",
            "Reasoning_std",
            "RewardBench Core_mean",
            "RewardBench Core_std",
        ]

        print(agg_df[display_cols].to_string(index=False))

        print("\nSaved per-seed summary to:")
        print(per_seed_summary_path)

        print("\nSaved mean/std summary to:")
        print(agg_path)

    else:
        print("\nNo model evaluation succeeded.")

    if all_failures:
        fail_df = pd.DataFrame(all_failures)

        fail_path = os.path.join(
            OUT_DIR,
            "failed_models_all_seeds.csv",
        )

        fail_df.to_csv(fail_path, index=False)

        print("\n================ FAILURES ================\n")
        print(fail_df.to_string(index=False))

        print("\nSaved failures to:")
        print(fail_path)

    print(f"\nSaved outputs to: {OUT_DIR}")
    print(f"Total wall-clock time: {round(time.time() - overall_start, 2)} sec")


if __name__ == "__main__":
    main()